In [ ]:
from typing import Callable, List, Tuple
from collections import OrderedDict

import os
import pandas as pd
import numpy as np
import time
from tqdm import tqdm
import cv2

import torch
from torch import nn, Tensor, no_grad
from torch import optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

PREPROCESSED_PATH = './data/preprocessed/'
os.makedirs(PREPROCESSED_PATH, exist_ok=True)
IMG_PATH = './data/images/'
os.makedirs(IMG_PATH, exist_ok=True)


device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

In [ ]:
df_all = pd.read_csv(os.path.join("raw_data", "cleaned_dataset.csv"))
df_all = df_all[(~df_all["Color"].isna()) & (~df_all["Color"].isin(["Gold", "Silver"]))]

def filter_existing_files(df: pd.DataFrame, path: str) -> pd.DataFrame:    
    return df[df["file_name"].apply(lambda file: os.path.exists(os.path.join(path, file)))]

df_all = filter_existing_files(df_all, PREPROCESSED_PATH)
df_all.shape

In [ ]:
def category_sample(df:pd.DataFrame, column:str, n:int) -> pd.DataFrame:
    return df.groupby(column).apply(lambda x: x.sample(n=min(n, len(x)))).reset_index(drop=True)

df = category_sample(df_all, "Color", 1500)
df.shape

In [ ]:
class ClothesDataset(Dataset):
    def __init__(self, 
            df:pd.DataFrame, 
            label_column: str,
            labels:List[str],
            transform:Callable[[np.ndarray], Tensor]=transforms.ToTensor(), 
            path_images:str="./data",
            filename_column: str='file_name', 
        ) -> None:
        if label_column not in df.columns:
            raise ValueError(f"label column '{label_column}' not found in DataFrame")      
        self.path_files = path_images
        self.label_column = label_column
        self.filename_column = filename_column
        self.labels = labels
        self.df = df.dropna(subset=[label_column])  # remove linhas sem categoria
        self.transform = transform
        
        self.parse_label = {}
        self.parse_label = {label: idx for idx, label in enumerate(labels)}

    def __len__(self) -> int:
        return len(self.df)
    
    def num_labels(self) -> int:
        return len(self.labels)
    
    def true_labels(self, indexed:bool=False) -> List[str]:
        if indexed:
            return self.df[self.label_column].apply(lambda x: self.parse_label[x]).tolist()
        return self.df[self.label_column].tolist()

    def __getitem__(self, idx:int) -> Tuple[Tensor, int]:
        row = self.df.iloc[idx]
        
        image_path = os.path.join(self.path_files, row[self.filename_column])
        if not os.path.isfile(image_path):
            raise FileNotFoundError(f"Image file '{image_path}' not found")
        
        image = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
        image = self.transform(image)
        label = self.parse_label[row[self.label_column]]

        return image, label

In [ ]:
image_size = (64, 64)

norm_mean = [0.485, 0.456, 0.406]
norm_std = [0.229, 0.224, 0.225]

transform = transforms.Compose([
  transforms.ToTensor(),
  transforms.Resize(image_size),
  transforms.Normalize(norm_mean, norm_std),
])

label_column = "Color"
labels = df_all[label_column].dropna().sort_values().unique().tolist()  
labels    

In [ ]:
df_train = df.sample(frac=0.8, random_state=42)
df_test = df.drop(df_train.index)

train_dataset = ClothesDataset(df_train, label_column, labels, transform=transform, path_images=PREPROCESSED_PATH)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)

test_dataset = ClothesDataset(df_test, label_column, labels, transform=transform, path_images=PREPROCESSED_PATH)
test_dataloader = DataLoader(test_dataset, batch_size=128, shuffle=False)

In [ ]:
class ColorCNN1(nn.Module): # accuracy: 66%
    def __init__(self, img_size: Tuple[int, int], num_labels: int):
        super(ColorCNN1, self).__init__()
        self.w, self.h = img_size
        self.num_labels = num_labels
        
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        final_w = self.w // 2 // 2
        final_h = self.h // 2 // 2
        self.fc1 = nn.Linear(64 * final_w * final_h, 256) 
        self.relu_fc1 = nn.ReLU()
        self.fc2 = nn.Linear(256, num_labels) 

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = x.view(x.size(0), -1) 
        x = self.relu_fc1(self.fc1(x))
        x = self.fc2(x)
        return x

class ColorCNN2(nn.Module): # accuracy: 75%
    def __init__(self, img_size: Tuple[int, int], num_labels: int):
        super(ColorCNN2, self).__init__()
        self.w, self.h = img_size
        self.num_labels = num_labels
        
        # Primeira camada convolucional: captura características básicas (bordas, texturas simples)
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Segunda camada convolucional: extrai características mais abstratas
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Terceira camada convolucional: aprofunda a análise da estrutura da imagem
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Camada totalmente conectada: transforma características extraídas em previsões
        final_w = self.w // 2 // 2 // 2 
        final_h = self.h // 2 // 2 // 2
        self.fc1 = nn.Linear(128 * final_w * final_h, 256) 
        self.relu_fc1 = nn.ReLU()
        self.fc2 = nn.Linear(256, num_labels) 

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.pool3(self.relu3(self.conv3(x)))
        x = x.view(x.size(0), -1) # Flatten
        x = self.relu_fc1(self.fc1(x))
        x = self.fc2(x)
        return x
    
class ColorCNN3(nn.Module): # accuracy: 74%
    def __init__(self, img_size: Tuple[int, int], num_labels: int):
        super(ColorCNN3, self).__init__()
        self.w, self.h = img_size
        self.num_labels = num_labels
        
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv4 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, stride=1, padding=1)
        self.relu4 = nn.ReLU()
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)

        final_w = self.w // 2 // 2 // 2 // 2 
        final_h = self.h // 2 // 2 // 2 // 2
        self.fc1 = nn.Linear(256 * final_w * final_h, 256)
        self.relu_fc1 = nn.ReLU()
        self.fc2 = nn.Linear(256, num_labels)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.pool3(self.relu3(self.conv3(x)))
        x = self.pool4(self.relu4(self.conv4(x)))
        x = x.view(x.size(0), -1) 
        x = self.relu_fc1(self.fc1(x))
        x = self.fc2(x)
        return x

model = ColorCNN2(image_size, train_dataset.num_labels())
model.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [ ]:
epochs = 10

running_losses = []

total_batches = epochs * len(train_dataloader)
progress_bar = tqdm(total=total_batches, desc="Training Progress", position=0, bar_format='{l_bar}{bar:20}{r_bar}')

start_time = time.time()
model.train()
for epoch in range(epochs):
    epoch_running_loss = 0
    for i, batch in enumerate(train_dataloader):
        imgs, labels = batch
        optimizer.zero_grad()
        
        output = model(imgs.to(device))
        
        loss = criterion(output, labels.to(device))
        loss.backward()
        optimizer.step()
        
        running_losses.append(loss.item())
        
        progress_bar.update(1)
        progress_bar.set_postfix(epoch=epoch+1, batch=i+1, loss=round(loss.item(), 2))
        
progress_bar.close()

In [ ]:
if not os.path.exists(f"./models"):
    os.makedirs(f"./models")

torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "time": time.time() - start_time,
        "running_losses": running_losses,
        "labels": labels,
        "image_size": image_size,
    }, 
    os.path.join("models", f"model_{label_column}.pt").lower()
)

torch.jit.trace(
    model,
    torch.randn(1, 3, image_size[0], image_size[1]).to(device)
).save(
    os.path.join("models", f"jit_model_{label_column}.pt").lower()
)

In [ ]:
correct = 0
pred_labels = []

total_batches = len(test_dataloader)
progress_bar = tqdm(total=total_batches, desc="Test Progress", position=0, bar_format='{l_bar}{bar:20}{r_bar}')

model.eval()
with no_grad():
    for i, batch in enumerate(test_dataloader):
        imgs, labels = batch
        prediction = model(imgs.to(device))
        _, prediction = torch.max(prediction.data, 1)
        correct += (prediction == labels.to(device)).sum().item()
        pred_labels += prediction.cpu().tolist()
        
        progress_bar.update(1)
        progress_bar.set_postfix(batch=i+1, accuracy=round(correct/len(pred_labels), 2))
    progress_bar.close()

In [ ]:
matrix = confusion_matrix(test_dataset.true_labels(True), pred_labels)
ConfusionMatrixDisplay(matrix, display_labels=test_dataset.labels).plot(xticks_rotation="vertical")